# cAPTure: corrected graph-logit development comparison

This notebook evaluates the four completed graph models using their immutable rescored raw logits. It repeats the frozen development-OOF alert-budget analysis and places the results beside the four previously validated XGBoost development benchmarks.

XGBoost is an external benchmark, not an architecture-matched ablation. No model is trained, no checkpoint is loaded, and no held-out scenario is accessed.

## 1. Mount Drive and prepare the CPU environment

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_logit_comparison import (
    run_graph_logit_comparison,
    validate_graph_logit_comparison,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("A GPU runtime is not required for this comparison.")

Mounted at /content/drive
Repository commit: e8f9efa1294eeee89efe208227094cf48572f298
A GPU runtime is not required for this comparison.


## 2. Bind the immutable graph and XGBoost evidence

In [2]:
RESCORING_RUN_ID = "20261001_graph_oof_logit_rescoring_v1"
SOURCE_GRAPH_COMPARISON_RUN_ID = (
    "20261001T153351_608797Z_graph_four_model_comparison"
)
SATURATION_AUDIT_RUN_ID = (
    "20261001T175306_830261Z_graph_score_saturation_audit"
)
XGB_OPERATIONAL_RUN_ID = (
    "20260923T153258_885662Z_operational_oof_v2_float64"
)
XGB_EARLY_WARNING_RUN_ID = (
    "20260923T155543_771684Z_early_warning_oof_v2_float64"
)
COMPARISON_RUN_ID = "20261001_graph_logit_comparison_v1"

RESCORING_ROOT = (
    DRIVE_ROOT / "graph_oof_rescoring_runs" / RESCORING_RUN_ID
)
SOURCE_GRAPH_COMPARISON_DIR = (
    DRIVE_ROOT / "graph_comparison_runs" / SOURCE_GRAPH_COMPARISON_RUN_ID
)
SATURATION_AUDIT_DIR = (
    DRIVE_ROOT / "graph_score_saturation_audit_runs"
    / SATURATION_AUDIT_RUN_ID
)
XGB_OPERATIONAL_DIR = (
    DRIVE_ROOT / "operational_oof_runs" / XGB_OPERATIONAL_RUN_ID
)
XGB_EARLY_WARNING_DIR = (
    DRIVE_ROOT / "early_warning_oof_runs" / XGB_EARLY_WARNING_RUN_ID
)
XGB_ABLATION_DIR = (
    DRIVE_ROOT / "xgb_p_t_ablation_runs"
    / "20260920T000745_916321Z_xgb_p_t_ablation"
)
XGB_RUN_DIRS = {
    "xgb_p": DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p",
    "current_window": XGB_ABLATION_DIR / "current_window",
    "history": XGB_ABLATION_DIR / "history",
    "full": DRIVE_ROOT / "xgb_p_t_runs/20260919T201909_754628Z_xgb_p_t",
}
OUTPUT_DIR = (
    DRIVE_ROOT / "graph_logit_comparison_runs" / COMPARISON_RUN_ID
)
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
TRAINING_CONFIG_PATH = PROJECT_ROOT / "configs/capture_graph_training_v1.yaml"
TERMINAL_POLICY_PATH = (
    PROJECT_ROOT / "configs/capture_early_warning_audit_v1.yaml"
)
RESCORING_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_oof_rescoring_v1.yaml"
)
COMPARISON_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_logit_comparison_v1.yaml"
)

required_paths = [
    RESCORING_ROOT, SOURCE_GRAPH_COMPARISON_DIR, SATURATION_AUDIT_DIR,
    XGB_OPERATIONAL_DIR, XGB_EARLY_WARNING_DIR, MANIFEST_PATH,
    TRAINING_CONFIG_PATH, TERMINAL_POLICY_PATH, RESCORING_CONFIG_PATH,
    COMPARISON_CONFIG_PATH, *XGB_RUN_DIRS.values(),
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")
print("Graph rescoring input:", RESCORING_ROOT)
print("XGB operational input:", XGB_OPERATIONAL_DIR)
print("Corrected comparison output:", OUTPUT_DIR)

Graph rescoring input: /content/drive/MyDrive/capture_gate0/graph_oof_rescoring_runs/20261001_graph_oof_logit_rescoring_v1
XGB operational input: /content/drive/MyDrive/capture_gate0/operational_oof_runs/20260923T153258_885662Z_operational_oof_v2_float64
Corrected comparison output: /content/drive/MyDrive/capture_gate0/graph_logit_comparison_runs/20261001_graph_logit_comparison_v1


## 3. Run or validate the corrected comparison

The evaluation reads large OOF Parquet files and can take several minutes on CPU. Set `RUN_COMPARISON=True` only after reviewing every bound path. A completed output is immutable and is checksum-validated on a rerun.

In [3]:
RUN_COMPARISON = True

comparison_arguments = {
    "comparison_config_path": COMPARISON_CONFIG_PATH,
    "rescoring_config_path": RESCORING_CONFIG_PATH,
    "rescoring_root": RESCORING_ROOT,
    "source_comparison_dir": SOURCE_GRAPH_COMPARISON_DIR,
    "saturation_audit_dir": SATURATION_AUDIT_DIR,
    "manifest_path": MANIFEST_PATH,
    "training_config_path": TRAINING_CONFIG_PATH,
    "terminal_policy_path": TERMINAL_POLICY_PATH,
    "xgb_operational_dir": XGB_OPERATIONAL_DIR,
    "xgb_early_warning_dir": XGB_EARLY_WARNING_DIR,
    "xgb_run_dirs": XGB_RUN_DIRS,
    "output_dir": OUTPUT_DIR,
}
if OUTPUT_DIR.exists():
    comparison_report = validate_graph_logit_comparison(**comparison_arguments)
elif RUN_COMPARISON:
    comparison_report = run_graph_logit_comparison(
        **comparison_arguments, batch_size=250_000
    )
else:
    comparison_report = None
    print("Comparison is disabled. No OOF scores were evaluated.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Evaluating raw-logit OOF predictions for edge_mlp...
Evaluating raw-logit OOF predictions for edge_gru...
Evaluating raw-logit OOF predictions for static_gnn...
Evaluating raw-logit OOF predictions for st_gnn...


## 4. Review the corrected metrics

In [4]:
if comparison_report is None:
    print("Run section 3 with RUN_COMPARISON=True before reviewing results.")
else:
    tables = comparison_report["tables"]
    print("Comparison status:", comparison_report["status"])
    print("Graph operational score field:",
          comparison_report["operational_graph_score_field"])
    print("Graph/XGB packet alignment:",
          comparison_report["aligned_graph_xgb_packet_counts"])
    display(pd.DataFrame(tables["combined_ranking_summary"]))
    display(pd.DataFrame(tables["combined_primary_operational_summary"]))
    display(pd.DataFrame(tables["graph_score_correction_impact"]))
    display(pd.DataFrame(tables["combined_budget_sensitivity_summary"]))
    display(pd.DataFrame(tables["pairwise_comparisons"]))
    display(pd.DataFrame(tables["graph_vs_xgb_benchmark_deltas"]))
    display(pd.DataFrame(tables["corrected_graph_resource_summary"]))
    display(Markdown((OUTPUT_DIR / "review.md").read_text(encoding="utf-8")))

Comparison status: exploratory_graph_logit_comparison_complete
Graph operational score field: raw_logit
Graph/XGB packet alignment: {'train_dollar_char': 2882555, 'train_slash_char': 6425516, 'train_sub_exf': 2225807, 'train_empty_conn': 1175779, 'train_qos_mid': 1499717}


,family,model,model_display,average_precision,roc_auc
0,graph,edge_mlp,Edge MLP,0.959440,0.958990
1,graph,edge_gru,EdgeGRU,0.995314,0.997029
2,graph,static_gnn,StaticGNN,0.994474,0.996179
3,graph,st_gnn,ST-GNN,0.989238,0.997217
4,xgb,xgb_p,XGB-P,0.968574,0.969854
5,xgb,current_window,XGB current window,0.998205,0.998623
6,xgb,history,XGB history,0.997909,0.998556
7,xgb,full,XGB full,0.996965,0.997555


,family,model,model_display,score_field,threshold,worst_fold_false_alert_windows_per_hour,false_alert_windows_per_hour,packet_recall,packet_precision,packet_false_positive_rate,packet_f1,packet_f2,sequence_detection_rate,timely_iteration_rate,early_before_terminal_action,timely_iterations_with_lead,median_seconds_before_iteration_end,early_chains_with_lead,median_seconds_before_terminal_action
0,graph,edge_mlp,Edge MLP,raw_logit,10.284599,0.000000,0.000000,0.273171,NaN,0.000000,0.353291,0.300421,0.170342,0.135268,0.500000,68,117.232691,2,4453.556418
1,graph,edge_gru,EdgeGRU,raw_logit,24.407442,0.984721,0.492360,0.263698,NaN,0.000011,0.331024,0.285673,0.302572,0.273291,0.500000,543,21.837653,3,3731.753496
2,graph,static_gnn,StaticGNN,raw_logit,7.059270,0.968959,0.957702,0.705229,0.999746,0.000074,0.808531,0.740819,0.740541,0.714158,1.000000,821,53.193836,5,3781.753496
3,graph,st_gnn,ST-GNN,raw_logit,17.958267,0.966029,0.483015,0.000078,NaN,0.000023,0.000155,0.000097,0.030063,0.022823,0.333333,33,9.784676,2,1278.841731
4,xgb,xgb_p,XGB-P,score,0.999463,0.417543,0.208772,0.581894,0.999995,0.000001,0.722528,0.629563,0.528501,0.395503,1.000000,548,10.285577,5,3731.753496
5,xgb,current_window,XGB current window,score,0.983387,0.798887,0.658179,0.827336,0.999797,0.000123,0.900890,0.854497,0.909566,0.847208,1.000000,1005,52.368274,5,3781.753496
6,xgb,history,XGB history,score,0.995854,0.979633,0.918791,0.815902,0.999954,0.000026,0.894119,0.844757,0.985104,0.884298,1.000000,1029,51.975060,5,3781.753496
7,xgb,full,XGB full,score,0.997506,0.866198,0.433099,0.771486,0.999886,0.000067,0.860757,0.803378,0.844987,0.827217,1.000000,996,52.262701,5,3781.753496


,model,model_display,old_float32_average_precision,raw_logit_average_precision,average_precision_delta,old_float32_packet_f1,raw_logit_packet_f1,old_float32_timely_iteration_rate,raw_logit_timely_iteration_rate,old_float32_worst_fold_false_alert_windows_per_hour,raw_logit_worst_fold_false_alert_windows_per_hour
0,edge_mlp,Edge MLP,0.959441,0.959440,-9.819452e-07,0.353291,0.353291,0.135268,0.135268,0.000000,0.000000
1,edge_gru,EdgeGRU,0.993317,0.995314,1.997118e-03,0.000000,0.331024,0.000000,0.273291,0.000000,0.984721
2,static_gnn,StaticGNN,0.994474,0.994474,-2.195776e-07,0.808529,0.808531,0.714158,0.714158,0.946445,0.968959
3,st_gnn,ST-GNN,0.992172,0.989238,-2.934167e-03,0.000000,0.000155,0.000000,0.022823,0.000000,0.966029


,family,model,model_display,budget,target_false_alert_windows_per_hour,threshold,worst_fold_false_alert_windows_per_hour,false_alert_windows_per_hour,packet_recall,packet_precision,packet_false_positive_rate,packet_f1,packet_f2,sequence_detection_rate,timely_iteration_rate,early_before_terminal_action,timely_iterations_with_lead,median_seconds_before_iteration_end,early_chains_with_lead,median_seconds_before_terminal_action
0,graph,edge_mlp,Edge MLP,one_per_hour,1.000000,10.284599,0.000000,0.000000,0.273171,NaN,0.000000e+00,0.353291,0.300421,0.170342,0.135268,0.500000,68,117.232691,2,4453.556418
1,graph,edge_mlp,Edge MLP,one_per_12_hours,0.083333,10.284599,0.000000,0.000000,0.273171,NaN,0.000000e+00,0.353291,0.300421,0.170342,0.135268,0.500000,68,117.232691,2,4453.556418
2,graph,edge_mlp,Edge MLP,one_per_5_minutes,12.000000,2.705220,9.615153,9.104160,0.711327,0.998008,1.040448e-03,0.817040,0.748848,1.000000,0.897643,1.000000,1056,52.243998,5,3781.753496
3,graph,edge_gru,EdgeGRU,one_per_hour,1.000000,24.407442,0.984721,0.492360,0.263698,NaN,1.055717e-05,0.331024,0.285673,0.302572,0.273291,0.500000,543,21.837653,3,3731.753496
4,graph,edge_gru,EdgeGRU,one_per_12_hours,0.083333,25.122387,0.072149,0.036074,0.251887,NaN,8.797638e-07,0.320421,0.274143,0.276390,0.246543,0.500000,493,18.870746,3,3731.753496
5,graph,edge_gru,EdgeGRU,one_per_5_minutes,12.000000,22.107710,11.914160,5.957080,0.297187,NaN,8.211129e-05,0.357377,0.316991,0.386394,0.372085,0.500000,697,32.903430,3,3731.753496
6,graph,static_gnn,StaticGNN,one_per_hour,1.000000,7.059270,0.968959,0.957702,0.705229,0.999746,7.446103e-05,0.808531,0.740819,0.740541,0.714158,1.000000,821,53.193836,5,3781.753496
7,graph,static_gnn,StaticGNN,one_per_12_hours,0.083333,14.641726,0.000000,0.000000,0.197105,NaN,0.000000e+00,0.275832,0.222190,0.187917,0.177161,0.500000,340,54.375649,3,3781.753496
8,graph,static_gnn,StaticGNN,one_per_5_minutes,12.000000,3.359082,11.970775,8.261523,0.793026,0.998293,7.903029e-04,0.865048,0.818163,0.877192,0.797004,1.000000,903,52.738365,5,3781.753496
9,graph,st_gnn,ST-GNN,one_per_hour,1.000000,17.958267,0.966029,0.483015,0.000078,NaN,2.346037e-05,0.000155,0.000097,0.030063,0.022823,0.333333,33,9.784676,2,1278.841731


,comparison,candidate,control,metric,candidate_hierarchical_macro,control_hierarchical_macro,hierarchical_macro_delta,scenarios_with_positive_delta,scenarios_with_zero_delta,scenarios_with_negative_delta,interpretation,frozen_practical_margin
0,temporal_memory_without_gat,edge_gru,edge_mlp,average_precision,0.995314,0.959440,0.035874,5,0,0,Incremental per-endpoint temporal memory witho...,0.005
1,temporal_memory_without_gat,edge_gru,edge_mlp,roc_auc,0.997029,0.958990,0.038039,5,0,0,Incremental per-endpoint temporal memory witho...,NaN
2,temporal_memory_without_gat,edge_gru,edge_mlp,packet_recall,0.263698,0.273171,-0.009473,3,0,2,Incremental per-endpoint temporal memory witho...,NaN
3,temporal_memory_without_gat,edge_gru,edge_mlp,packet_f1,0.331024,0.353291,-0.022266,3,0,2,Incremental per-endpoint temporal memory witho...,NaN
4,temporal_memory_without_gat,edge_gru,edge_mlp,packet_f2,0.285673,0.300421,-0.014748,3,0,2,Incremental per-endpoint temporal memory witho...,NaN
5,temporal_memory_without_gat,edge_gru,edge_mlp,sequence_detection_rate,0.302572,0.170342,0.132230,3,0,2,Incremental per-endpoint temporal memory witho...,NaN
6,temporal_memory_without_gat,edge_gru,edge_mlp,timely_iteration_rate,0.273291,0.135268,0.138023,3,0,2,Incremental per-endpoint temporal memory witho...,0.020
7,temporal_memory_without_gat,edge_gru,edge_mlp,early_before_terminal_action,0.500000,0.500000,0.000000,3,0,2,Incremental per-endpoint temporal memory witho...,NaN
8,current_message_passing_without_memory,static_gnn,edge_mlp,average_precision,0.994474,0.959440,0.035034,5,0,0,Current endpoint aggregation and GAT message p...,0.005
9,current_message_passing_without_memory,static_gnn,edge_mlp,roc_auc,0.996179,0.958990,0.037188,5,0,0,Current endpoint aggregation and GAT message p...,NaN


,graph_model,graph_model_display,xgb_reference,xgb_reference_display,metric,graph_hierarchical_macro,xgb_hierarchical_macro,graph_minus_xgb_delta,scenarios_with_positive_delta,scenarios_with_zero_delta,scenarios_with_negative_delta,causal_architecture_contrast
0,edge_mlp,Edge MLP,xgb_p,XGB-P,average_precision,0.959440,0.968574,-0.009134,0,0,5,False
1,edge_mlp,Edge MLP,xgb_p,XGB-P,roc_auc,0.958990,0.969854,-0.010864,0,0,5,False
2,edge_mlp,Edge MLP,xgb_p,XGB-P,packet_recall,0.273171,0.581894,-0.308723,0,0,5,False
3,edge_mlp,Edge MLP,xgb_p,XGB-P,packet_f1,0.353291,0.722528,-0.369237,0,0,5,False
4,edge_mlp,Edge MLP,xgb_p,XGB-P,packet_f2,0.300421,0.629563,-0.329142,0,0,5,False
...,...,...,...,...,...,...,...,...,...,...,...,...
59,st_gnn,ST-GNN,history,XGB history,packet_f1,0.000155,0.894119,-0.893964,0,0,5,False
60,st_gnn,ST-GNN,history,XGB history,packet_f2,0.000097,0.844757,-0.844660,0,0,5,False
61,st_gnn,ST-GNN,history,XGB history,sequence_detection_rate,0.030063,0.985104,-0.955041,0,0,5,False
62,st_gnn,ST-GNN,history,XGB history,timely_iteration_rate,0.022823,0.884298,-0.861475,0,0,5,False


,model,model_display,fold_jobs,best_epoch_count_fold_A,best_epoch_count_fold_B,parameters,total_wall_seconds,maximum_peak_host_rss_gib,maximum_peak_cuda_allocated_gib,oof_packets,pure_inference_seconds,seconds_per_million_packets,source_training_and_original_job_wall_seconds,corrected_oof_inference_seconds,corrected_oof_seconds_per_million_packets
0,edge_mlp,Edge MLP,2,1,21,11137,4172.588271,2.099953,0.118065,14209374,74.987401,5.277319,4172.588271,80.821302,5.687886
1,edge_gru,EdgeGRU,2,11,1,50946,5824.838758,2.318100,0.190306,14209374,153.606470,10.810221,5824.838758,161.200323,11.344646
2,static_gnn,StaticGNN,2,12,3,55745,9467.906064,2.313602,0.273281,14209374,258.599590,18.199225,9467.906064,247.111088,17.390709
3,st_gnn,ST-GNN,2,3,3,80706,9254.983488,2.327396,0.275872,14209374,305.259083,21.482937,9254.983488,307.310133,21.627282


# cAPTure corrected graph-logit development comparison

Status: **exploratory corrected comparison complete**.

This report replaces saturated float32 graph probabilities with raw logits for ranking and operational thresholding. It uses only checksum-verified seed-42 development OOF predictions. No checkpoint is loaded and no model is trained.

## Corrected ranking metrics and XGB benchmarks

| Family | Model | Hierarchical AP | Hierarchical ROC-AUC |
|---|---|---|---|
| graph | Edge MLP | 0.959440 | 0.958990 |
| graph | EdgeGRU | 0.995314 | 0.997029 |
| graph | StaticGNN | 0.994474 | 0.996179 |
| graph | ST-GNN | 0.989238 | 0.997217 |
| xgb | XGB-P | 0.968574 | 0.969854 |
| xgb | XGB current window | 0.998205 | 0.998623 |
| xgb | XGB history | 0.997909 | 0.998556 |
| xgb | XGB full | 0.996965 | 0.997555 |

## Primary operational budget

Each model retains its own development-OOF threshold under the frozen one-false-alert-window-per-hour budget. XGB is an external development benchmark, not an architecture-matched ablation.

| Family | Model | Worst-fold FA/h | Precision | Recall | F1 | F2 | Iteration coverage | Timely coverage | Preterminal coverage |
|---|---|---|---|---|---|---|---|---|---|
| graph | Edge MLP | 0.0000 | NA | 0.2732 | 0.3533 | 0.3004 | 0.1703 | 0.1353 | 0.5000 |
| graph | EdgeGRU | 0.9847 | NA | 0.2637 | 0.3310 | 0.2857 | 0.3026 | 0.2733 | 0.5000 |
| graph | StaticGNN | 0.9690 | 0.9997 | 0.7052 | 0.8085 | 0.7408 | 0.7405 | 0.7142 | 1.0000 |
| graph | ST-GNN | 0.9660 | NA | 0.0001 | 0.0002 | 0.0001 | 0.0301 | 0.0228 | 0.3333 |
| xgb | XGB-P | 0.4175 | 1.0000 | 0.5819 | 0.7225 | 0.6296 | 0.5285 | 0.3955 | 1.0000 |
| xgb | XGB current window | 0.7989 | 0.9998 | 0.8273 | 0.9009 | 0.8545 | 0.9096 | 0.8472 | 1.0000 |
| xgb | XGB history | 0.9796 | 1.0000 | 0.8159 | 0.8941 | 0.8448 | 0.9851 | 0.8843 | 1.0000 |
| xgb | XGB full | 0.8662 | 0.9999 | 0.7715 | 0.8608 | 0.8034 | 0.8450 | 0.8272 | 1.0000 |

## Effect of replacing stored float32 probabilities

| Model | Old AP | Logit AP | Old F1 | Logit F1 | Old timely | Logit timely |
|---|---|---|---|---|---|---|
| Edge MLP | 0.9594 | 0.9594 | 0.3533 | 0.3533 | 0.1353 | 0.1353 |
| EdgeGRU | 0.9933 | 0.9953 | 0.0000 | 0.3310 | 0.0000 | 0.2733 |
| StaticGNN | 0.9945 | 0.9945 | 0.8085 | 0.8085 | 0.7142 | 0.7142 |
| ST-GNN | 0.9922 | 0.9892 | 0.0000 | 0.0002 | 0.0000 | 0.0228 |

## Predeclared graph contrasts

| Comparison | Metric | Delta | Positive scenarios | Negative scenarios |
|---|---|---|---|---|
| temporal_memory_without_gat | average_precision | 0.035874 | 5 | 0 |
| temporal_memory_without_gat | timely_iteration_rate | 0.138023 | 3 | 2 |
| current_message_passing_without_memory | average_precision | 0.035034 | 5 | 0 |
| current_message_passing_without_memory | timely_iteration_rate | 0.578890 | 5 | 0 |
| temporal_memory_with_gat | average_precision | -0.005236 | 2 | 3 |
| temporal_memory_with_gat | timely_iteration_rate | -0.691335 | 0 | 5 |
| gat_signal_in_temporal_models | average_precision | -0.006076 | 2 | 3 |
| gat_signal_in_temporal_models | timely_iteration_rate | -0.250468 | 0 | 3 |

## Interpretation boundaries

- Graph operational results in this report supersede the saturated operational rows in the original four-model comparison.
- XGB and graph models share packets, folds, windows, labels, alert budgets, and evaluation rules, but they do not share an architecture or identical feature representation. Their differences are benchmarking evidence, not causal ablations.
- `StaticGNN - Edge MLP` is the current message-passing signal without recurrent memory.
- `ST-GNN - EdgeGRU` remains only a preliminary temporal message-passing contrast.
- The matched ST-GNN control without GAT message passing, the no-direct-edge-attribute control, and shifted-origin sensitivity remain pending.
- One seed and reused development OOF calibration make every result exploratory.

Test1, Test2, `train_pub_exf`, and `train_user_prop` were not accessed.


## 5. Provenance boundary

In [5]:
if comparison_report is not None:
    print("Model training performed:",
          comparison_report["model_training_performed"])
    print("Checkpoint loading performed:",
          comparison_report["checkpoint_loading_performed"])
    print("Held-out scenarios accessed:",
          comparison_report["held_out_scenarios_accessed"])
    print("XGB treated as architecture ablation:",
          not comparison_report["xgb_is_external_development_benchmark_not_architecture_ablation"])
    print("Saved report:", OUTPUT_DIR / "comparison_report.json")
    print("Saved review:", OUTPUT_DIR / "review.md")

Model training performed: False
Checkpoint loading performed: False
Held-out scenarios accessed: False
XGB treated as architecture ablation: False
Saved report: /content/drive/MyDrive/capture_gate0/graph_logit_comparison_runs/20261001_graph_logit_comparison_v1/comparison_report.json
Saved review: /content/drive/MyDrive/capture_gate0/graph_logit_comparison_runs/20261001_graph_logit_comparison_v1/review.md
